In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — RQ4: Classical ML Models (5 models, 4 families)
# Input : tfidf_vectorizer.pkl + train/val/test.csv (text_cleaned_classical)
# Models: NB | LogReg | Linear SVM | Random Forest | XGBoost
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print("✓ Reproducibility block applied (SEED=42)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR    = "/content/drive/MyDrive/NLP FINAL"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
STAGE_DIR   = os.path.join(BASE_DIR, "rq4_classical_ml")
RQ3_MODELS  = os.path.join(BASE_DIR, "rq3_feature_engineering", "models")

MODELS_DIR   = os.path.join(STAGE_DIR, "models")
RESULTS_DIR  = os.path.join(STAGE_DIR, "results")
LOGS_DIR     = os.path.join(STAGE_DIR, "logs")
DIAGRAMS_DIR = os.path.join(STAGE_DIR, "diagrams")

for folder in [MODELS_DIR, RESULTS_DIR, LOGS_DIR, DIAGRAMS_DIR]:
    os.makedirs(folder, exist_ok=True)

TRAIN_PATH = os.path.join(DATASET_DIR, "train.csv")
VAL_PATH   = os.path.join(DATASET_DIR, "val.csv")
TEST_PATH  = os.path.join(DATASET_DIR, "test.csv")
TFIDF_PATH = os.path.join(RQ3_MODELS, "tfidf_vectorizer.pkl")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, TFIDF_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}\nRun earlier stages first.")

print(f"✓ STAGE_DIR : {STAGE_DIR}")


# ═══ CELL 3 — Install + imports ═══
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "scikit-learn", "xgboost", "joblib", "matplotlib", "seaborn"])

import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import platform
from datetime import datetime
from scipy.special import expit

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, auc,
)

DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = DPI
plt.rcParams["savefig.dpi"] = DPI


# ═══ CELL 4 — Load data + TF-IDF (NEVER refit vectorizer) ═══
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

tfidf_vectorizer = joblib.load(TFIDF_PATH)   # load only — never .fit()

X_train = tfidf_vectorizer.transform(train_df["text_cleaned_classical"].astype(str))
X_val   = tfidf_vectorizer.transform(val_df["text_cleaned_classical"].astype(str))
X_test  = tfidf_vectorizer.transform(test_df["text_cleaned_classical"].astype(str))

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

n_safe     = int((y_train == 0).sum())
n_phishing = int((y_train == 1).sum())
scale_pos_weight = round(n_safe / n_phishing, 6)

print(f"Train : {X_train.shape[0]:,} samples × {X_train.shape[1]:,} TF-IDF features")
print(f"Val   : {X_val.shape[0]:,} | Test : {X_test.shape[0]:,}")
print(f"XGBoost scale_pos_weight = {scale_pos_weight}")


# ═══ CELL 5 — Model definitions (RQ2 imbalance handling applied) ═══
MODEL_CONFIGS = {
    "naive_bayes": {
        "display_name": "Multinomial Naive Bayes",
        "family": "Probabilistic",
        "estimator": MultinomialNB(class_prior=[0.5, 0.5], fit_prior=False),
    },
    "logistic_regression": {
        "display_name": "Logistic Regression",
        "family": "Linear",
        "estimator": LogisticRegression(
            class_weight="balanced", random_state=SEED, max_iter=1000, solver="lbfgs"
        ),
    },
    "linear_svm": {
        "display_name": "Linear SVM",
        "family": "Margin-based",
        "estimator": LinearSVC(
            class_weight="balanced", random_state=SEED, max_iter=3000, dual="auto"
        ),
    },
    "random_forest": {
        "display_name": "Random Forest",
        "family": "Tree ensemble (bagging)",
        "estimator": RandomForestClassifier(
            class_weight="balanced", random_state=SEED, n_estimators=200, n_jobs=-1
        ),
    },
    "xgboost": {
        "display_name": "XGBoost",
        "family": "Tree ensemble (boosting)",
        "estimator": XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            random_state=SEED,
            eval_metric="logloss",
            verbosity=0,
            n_estimators=200,
            n_jobs=-1,
        ),
    },
}


# ═══ CELL 6 — Helper functions ═══
def get_phishing_scores(model, X):
    """Return P(phishing) or equivalent score in [0,1] for ROC/PR."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return expit(model.decision_function(X))
    raise AttributeError("Model has no predict_proba or decision_function")


def compute_metrics(y_true, y_pred, y_score):
    return {
        "accuracy":        round(accuracy_score(y_true, y_pred), 4),
        "precision":       round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":          round(recall_score(y_true, y_pred, zero_division=0), 4),
        "macro_f1":        round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "roc_auc":         round(roc_auc_score(y_true, y_score), 4),
        "pr_auc":          round(average_precision_score(y_true, y_score), 4),
        "phishing_recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }


def save_confusion_matrix_fig(cm, title, filepath, normalize=False):
    if normalize:
        cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt, vmax = ".2f", 1.0
        suffix = " (Normalized)"
    else:
        cm_plot, fmt, vmax = cm, "d", None
        suffix = " (Raw Counts)"

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(
        cm_plot, annot=True, fmt=fmt, cmap="Blues", vmax=vmax,
        xticklabels=["Legitimate", "Phishing"],
        yticklabels=["Legitimate", "Phishing"],
        ax=ax, linewidths=0.5, linecolor="gray",
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title + suffix, fontweight="bold")
    plt.tight_layout()
    plt.savefig(filepath, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close()


def save_roc_fig(y_true, y_score, title, filepath):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc_val = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="#e74c3c", lw=2, label=f"ROC AUC = {roc_auc_val:.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title, fontweight="bold")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(filepath, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close()
    return roc_auc_val


# ═══ CELL 7 — Train, evaluate, save (all 5 models) ═══
print("\nTraining & evaluating 5 classical models on TEST set...\n")

overall_metrics_rows = []
classification_rows  = []
confusion_rows       = []
test_pred_df         = pd.DataFrame({"y_true": y_test})
roc_overlay_data     = []

for model_key, cfg in MODEL_CONFIGS.items():
    name = cfg["display_name"]
    print(f"── {name} ──")

    model = cfg["estimator"]
    model.fit(X_train, y_train)

    model_path = os.path.join(MODELS_DIR, f"{model_key}.pkl")
    joblib.dump(model, model_path)
    print(f"  ✓ Model saved → {model_key}.pkl")

    y_pred  = model.predict(X_test)
    y_score = get_phishing_scores(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    cm      = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"  Accuracy={metrics['accuracy']} | Macro-F1={metrics['macro_f1']} | "
          f"ROC-AUC={metrics['roc_auc']} | Phishing-Recall={metrics['phishing_recall']}")

    overall_metrics_rows.append({
        "model_key": model_key,
        "model_name": name,
        "model_family": cfg["family"],
        "eval_set": "test",
        **metrics,
    })

    report_dict = classification_report(
        y_test, y_pred, target_names=["Legitimate", "Phishing"],
        output_dict=True, zero_division=0,
    )
    for label_name in ["Legitimate", "Phishing"]:
        classification_rows.append({
            "model_key": model_key,
            "model_name": name,
            "class": label_name,
            "precision": round(report_dict[label_name]["precision"], 4),
            "recall":    round(report_dict[label_name]["recall"], 4),
            "f1_score":  round(report_dict[label_name]["f1-score"], 4),
            "support":   int(report_dict[label_name]["support"]),
        })
    classification_rows.append({
        "model_key": model_key,
        "model_name": name,
        "class": "macro avg",
        "precision": round(report_dict["macro avg"]["precision"], 4),
        "recall":    round(report_dict["macro avg"]["recall"], 4),
        "f1_score":  round(report_dict["macro avg"]["f1-score"], 4),
        "support":   int(report_dict["macro avg"]["support"]),
    })

    confusion_rows.append({
        "model_key": model_key,
        "model_name": name,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "specificity": round(tn / (tn + fp), 4) if (tn + fp) else 0,
        "phishing_recall": metrics["phishing_recall"],
    })

    test_pred_df[f"{model_key}_y_pred"]        = y_pred
    test_pred_df[f"{model_key}_phishing_prob"] = np.round(y_score, 6)

    save_confusion_matrix_fig(
        cm, name,
        os.path.join(DIAGRAMS_DIR, f"rq4_cm_{model_key}_raw.png"),
        normalize=False,
    )
    save_confusion_matrix_fig(
        cm, name,
        os.path.join(DIAGRAMS_DIR, f"rq4_cm_{model_key}_normalized.png"),
        normalize=True,
    )
    save_roc_fig(
        y_test, y_score, f"ROC Curve — {name}",
        os.path.join(DIAGRAMS_DIR, f"rq4_roc_{model_key}.png"),
    )

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_overlay_data.append((name, fpr, tpr, metrics["roc_auc"]))
    print()


# ═══ CELL 8 — Combined ROC overlay (all 5 models) ═══
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6", "#f39c12"]
for (name, fpr, tpr, roc_val), color in zip(roc_overlay_data, colors):
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name} ({roc_val:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("RQ4 — ROC Curves (All Classical Models)", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(
    os.path.join(DIAGRAMS_DIR, "rq4_roc_all_models_overlay.png"),
    dpi=DPI, bbox_inches="tight", facecolor="white",
)
plt.close()
print("✓ Saved combined ROC overlay")


# ═══ CELL 9 — Save results CSVs ═══
overall_metrics_df = pd.DataFrame(overall_metrics_rows)
overall_metrics_df.to_csv(os.path.join(RESULTS_DIR, "overall_metrics.csv"), index=False)

pd.DataFrame(classification_rows).to_csv(
    os.path.join(RESULTS_DIR, "classification_report.csv"), index=False
)
pd.DataFrame(confusion_rows).to_csv(
    os.path.join(RESULTS_DIR, "confusion_matrix.csv"), index=False
)
test_pred_df.to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)

# Paper-ready table — sort BEFORE renaming columns (fixes KeyError)
paper_table = overall_metrics_df[[
    "model_name", "model_family", "accuracy", "precision", "recall",
    "macro_f1", "roc_auc", "pr_auc", "phishing_recall",
]].copy()

paper_table = paper_table.sort_values("macro_f1", ascending=False).reset_index(drop=True)

paper_table.columns = [
    "Model", "Family", "Accuracy", "Precision", "Recall",
    "Macro-F1", "ROC-AUC", "PR-AUC", "Phishing Recall",
]

paper_table.to_csv(os.path.join(RESULTS_DIR, "paper_table_rq4_classical.csv"), index=False)

print("✓ Saved results CSVs:")
for f in [
    "overall_metrics.csv", "classification_report.csv",
    "confusion_matrix.csv", "test_predictions.csv",
    "paper_table_rq4_classical.csv",
]:
    print(f"    {f}")


# ═══ CELL 10 — experiment_config.csv ═══
config = {
    "project":                  "PhishGuard AI",
    "stage":                    "RQ4 — Classical ML Models",
    "timestamp_utc":            datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed":                     SEED,
    "text_column":              "text_cleaned_classical",
    "vectorizer":               TFIDF_PATH,
    "vectorizer_refit":         False,
    "train_samples":            X_train.shape[0],
    "test_samples":             X_test.shape[0],
    "tfidf_features":           X_train.shape[1],
    "models_trained":           ", ".join(MODEL_CONFIGS.keys()),
    "nb_strategy":            "class_prior=[0.5,0.5], fit_prior=False",
    "logreg_svm_rf_strategy": "class_weight='balanced'",
    "xgboost_scale_pos_weight": scale_pos_weight,
    "eval_set":                 "test",
    "diagram_dpi":              DPI,
    "python_version":           platform.python_version(),
    "sklearn_version":          __import__("sklearn").__version__,
    "xgboost_version":          __import__("xgboost").__version__,
}
pd.DataFrame(list(config.items()), columns=["parameter", "value"]).to_csv(
    os.path.join(RESULTS_DIR, "experiment_config.csv"), index=False
)


# ═══ CELL 11 — Final summary ═══
print("\n" + "=" * 70)
print("RQ4 — CLASSICAL ML COMPLETE ✓")
print("=" * 70)
print(f"\nModels saved → {MODELS_DIR}/")
print("  naive_bayes.pkl | logistic_regression.pkl | linear_svm.pkl")
print("  random_forest.pkl | xgboost.pkl")
print(f"\nDiagrams saved → {DIAGRAMS_DIR}/  (10 CM + 5 ROC + 1 overlay)")
print(f"\nResults saved → {RESULTS_DIR}/")
print("\n── Test Set Performance (sorted by Macro-F1) ──")
display(paper_table)
print("\nNext step → RQ5 (BiLSTM + TextCNN)")
print("=" * 70)

✓ Reproducibility block applied (SEED=42)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ STAGE_DIR : /content/drive/MyDrive/NLP FINAL/rq4_classical_ml
Train : 13,041 samples × 10,000 TF-IDF features
Val   : 2,795 | Test : 2,795
XGBoost scale_pos_weight = 1.549062

Training & evaluating 5 classical models on TEST set...

── Multinomial Naive Bayes ──
  ✓ Model saved → naive_bayes.pkl
  Accuracy=0.9567 | Macro-F1=0.9549 | ROC-AUC=0.9916 | Phishing-Recall=0.9644

── Logistic Regression ──
  ✓ Model saved → logistic_regression.pkl
  Accuracy=0.9664 | Macro-F1=0.965 | ROC-AUC=0.9923 | Phishing-Recall=0.9772

── Linear SVM ──
  ✓ Model saved → linear_svm.pkl
  Accuracy=0.9699 | Macro-F1=0.9687 | ROC-AUC=0.9945 | Phishing-Recall=0.9827

── Random Forest ──
  ✓ Model saved → random_forest.pkl
  Accuracy=0.9606 | Macro-F1=0.9589 | ROC-AUC=0.9924 | Phishing-Recall=0.9626

── XGBoost ──
  ✓ Model saved → xgboost.p

/tmp/ipykernel_1069/2757312726.py:370: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc":            datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


,Model,Family,Accuracy,Precision,Recall,Macro-F1,ROC-AUC,PR-AUC,Phishing Recall
0,Linear SVM,Margin-based,0.9699,0.9431,0.9827,0.9687,0.9945,0.9912,0.9827
1,Logistic Regression,Linear,0.9664,0.9395,0.9772,0.9650,0.9923,0.9867,0.9772
2,Random Forest,Tree ensemble (bagging),0.9606,0.9386,0.9626,0.9589,0.9924,0.9871,0.9626
3,XGBoost,Tree ensemble (boosting),0.9574,0.9244,0.9708,0.9557,0.9927,0.9865,0.9708
4,Multinomial Naive Bayes,Probabilistic,0.9567,0.9280,0.9644,0.9549,0.9916,0.9850,0.9644



Next step → RQ5 (BiLSTM + TextCNN)
